# Stops: mi perfil de ubicaciones antes de Silver

Revisé estas consultas mediante Genie el **10 de noviembre de 2026** y recuperé sus resultados estructurados. Conservo aquí las seis consultas verificadas como una sola fuente de lectura; sus resultados agregados están en [el informe](../../../docs/stops_profile_2025_2026.md).

Este notebook recién preparado no tiene una ejecución completa documentada en Databricks. Sus consultas sí se ejecutaron por separado mediante Genie. No necesito repetirlas para estudiar sus resultados.

Trabajo con los snapshots **2025-10-28** y **2026-09-29**, identificados por sus hashes. Los nombres `workspace.gtfs_bogota` son los valores reales de este ejemplo, no configuración universal. Este cuaderno es evidencia y exploración fijada a esas versiones; la configuración reutilizable de publicación pertenece al notebook 10.

Todas las celdas son SQL de lectura: no instalan librerías, no importan módulos Python, no crean tablas ni forman parte del Job. Puedo leer los bloques en orden y ejecutar una consulta aislada. El Job Silver sigue usando el notebook 10; stops tiene procesador implementado localmente; su publicación en Databricks sigue pendiente.


## Bloque 1 — ¿Qué columnas tengo y cómo están almacenadas?

El esquema histórico tiene 13 columnas: nueve de negocio y cuatro de procedencia. Los campos de negocio siguen como string. Que wheelchair_boarding aparezca aquí no demuestra que existiera en el archivo de 2025.


In [ ]:
%sql
-- BLOQUE 1 — ¿QUÉ COLUMNAS TENGO Y CÓMO ESTÁN ALMACENADAS? — Consulta de lectura ejecutada mediante Genie.
SELECT 
  `column_name`,
  `data_type`,
  `is_nullable`,
  `ordinal_position`
FROM workspace.information_schema.columns
WHERE `table_catalog` = 'workspace'
  AND `table_schema` = 'gtfs_bogota'
  AND `table_name` = 'bronze_gtfs_stops'
ORDER BY `ordinal_position`;


## Bloque 2 — ¿Cuántas ubicaciones e IDs hay por snapshot?

Encontré 6.530 y 8.314 filas, con igual cantidad de stop_id distintos. Esta consulta inicial identifica los hashes disponibles por fecha; las consultas siguientes fijan las dos parejas exactas. La diferencia de filas no demuestra expansión física del sistema ni continuidad de IDs entre años.


In [ ]:
%sql
-- BLOQUE 2 — ¿CUÁNTAS UBICACIONES E IDS HAY POR SNAPSHOT? — Consulta de lectura ejecutada mediante Genie.
SELECT 
  `source_feed_version`,
  `source_sha256`,
  COUNT(*) AS `total_rows`,
  COUNT(DISTINCT `stop_id`) AS `distinct_stop_ids`
FROM `workspace`.`gtfs_bogota`.`bronze_gtfs_stops`
WHERE `source_feed_version` IN ('2025-10-28', '2026-09-29')
GROUP BY `source_feed_version`, `source_sha256`
ORDER BY `source_feed_version`;


## Bloque 3 — ¿Qué columnas publicó cada archivo?

La auditoría contiene ocho nombres de columnas para 2025 y nueve para 2026. Wheelchair_boarding se añadió en 2026. Consulto los conjuntos de nombres registrados; no deduzco el encabezado por los null de Bronze ni afirmo releer el TXT desde Genie.


In [ ]:
%sql
-- BLOQUE 3 — ¿QUÉ COLUMNAS PUBLICÓ CADA ARCHIVO? — Consulta de lectura ejecutada mediante Genie.
SELECT DISTINCT
  `source_feed_version`,
  `source_sha256`,
  `source_file_name`,
  `source_columns_json`
FROM `workspace`.`gtfs_bogota`.`audit_gtfs_source_schema_observations`
WHERE `source_file_name` = 'stops.txt'
  AND (
    (`source_feed_version` = '2025-10-28' AND `source_sha256` = '07ba2ba9bb3b5bc1e38357355956757ce84fc68e6d05b74577bf017e189ab952')
    OR
    (`source_feed_version` = '2026-09-29' AND `source_sha256` = '08d14e0b3a5b9caa5451b722659871fd554e3273b94bdd770fdf81a26bf02097')
  )
ORDER BY `source_feed_version`;


## Bloque 4 — ¿Todas las filas representan lo mismo?

En 2025 todas las ubicaciones tienen location_type=0. En 2026 hay 8.159 paradas/plataformas de tipo 0 y 155 estaciones de tipo 1. Wheelchair_boarding vale 1 en todas las filas de 2026; es información publicada, no una inspección física de accesibilidad. Zone_id es una zona tarifaria. En esta salida [NULL] es solo una etiqueta de visualización; no se escribe ese texto en los datos. La consulta devolvió ocho grupos, menos que su límite de 50.


In [ ]:
%sql
-- BLOQUE 4 — ¿TODAS LAS FILAS REPRESENTAN LO MISMO? — Consulta de lectura ejecutada mediante Genie.
WITH location_type_dist AS (
  SELECT 
    `source_feed_version`,
    `source_sha256`,
    'location_type' AS field_name,
    COALESCE(`location_type`, '[NULL]') AS field_value,
    COUNT(*) AS value_count
  FROM `workspace`.`gtfs_bogota`.`bronze_gtfs_stops`
  WHERE (`source_feed_version` = '2025-10-28' AND `source_sha256` = '07ba2ba9bb3b5bc1e38357355956757ce84fc68e6d05b74577bf017e189ab952')
     OR (`source_feed_version` = '2026-09-29' AND `source_sha256` = '08d14e0b3a5b9caa5451b722659871fd554e3273b94bdd770fdf81a26bf02097')
  GROUP BY `source_feed_version`, `source_sha256`, COALESCE(`location_type`, '[NULL]')
),
wheelchair_boarding_dist AS (
  SELECT 
    `source_feed_version`,
    `source_sha256`,
    'wheelchair_boarding' AS field_name,
    COALESCE(`wheelchair_boarding`, '[NULL]') AS field_value,
    COUNT(*) AS value_count
  FROM `workspace`.`gtfs_bogota`.`bronze_gtfs_stops`
  WHERE (`source_feed_version` = '2025-10-28' AND `source_sha256` = '07ba2ba9bb3b5bc1e38357355956757ce84fc68e6d05b74577bf017e189ab952')
     OR (`source_feed_version` = '2026-09-29' AND `source_sha256` = '08d14e0b3a5b9caa5451b722659871fd554e3273b94bdd770fdf81a26bf02097')
  GROUP BY `source_feed_version`, `source_sha256`, COALESCE(`wheelchair_boarding`, '[NULL]')
),
zone_id_dist AS (
  SELECT 
    `source_feed_version`,
    `source_sha256`,
    'zone_id' AS field_name,
    COALESCE(`zone_id`, '[NULL]') AS field_value,
    COUNT(*) AS value_count
  FROM `workspace`.`gtfs_bogota`.`bronze_gtfs_stops`
  WHERE (`source_feed_version` = '2025-10-28' AND `source_sha256` = '07ba2ba9bb3b5bc1e38357355956757ce84fc68e6d05b74577bf017e189ab952')
     OR (`source_feed_version` = '2026-09-29' AND `source_sha256` = '08d14e0b3a5b9caa5451b722659871fd554e3273b94bdd770fdf81a26bf02097')
  GROUP BY `source_feed_version`, `source_sha256`, COALESCE(`zone_id`, '[NULL]')
),
combined AS (
  SELECT * FROM location_type_dist
  UNION ALL SELECT * FROM wheelchair_boarding_dist
  UNION ALL SELECT * FROM zone_id_dist
)
SELECT * FROM combined
ORDER BY `source_feed_version`, field_name, value_count DESC
LIMIT 50;


## Bloque 5 — ¿Dónde faltan valores o hay espacios?

Distingo nulo, cadena vacía, solo espacios y contenido. Esas cuatro categorías explican todas las filas de cada campo. Los distintos y los espacios en extremos son diagnósticos adicionales. Encontré 4 stop_code nulos en 2025 y 159 en 2026; stop_code es opcional y no es la clave. Hay 27 nombres con espacios extremos en 2025 y uno en 2026. Esta consulta usa Unicode (?U) y una agregación por campo en lugar de nueve CTEs repetidos.


In [ ]:
%sql
-- BLOQUE 5 — ¿DÓNDE FALTAN VALORES O HAY ESPACIOS? — Consulta de lectura ejecutada mediante Genie.
with base as (
  select source_feed_version, source_sha256, stop_id, stop_code, stop_name, stop_lat, stop_lon, location_type, parent_station, zone_id, wheelchair_boarding, source_file_path, bronze_loaded_at_utc
  from workspace.gtfs_bogota.bronze_gtfs_stops
  where (source_feed_version = '2025-10-28' and source_sha256 = '07ba2ba9bb3b5bc1e38357355956757ce84fc68e6d05b74577bf017e189ab952')
    or (source_feed_version = '2026-09-29' and source_sha256 = '08d14e0b3a5b9caa5451b722659871fd554e3273b94bdd770fdf81a26bf02097')
), cells as (
  select source_feed_version, source_sha256,
    stack(9, 'stop_id', stop_id, 'stop_code', stop_code, 'stop_name', stop_name, 'stop_lat', stop_lat, 'stop_lon', stop_lon, 'location_type', location_type, 'parent_station', parent_station, 'zone_id', zone_id, 'wheelchair_boarding', wheelchair_boarding) as (field_name, field_value)
  from base
)
select source_feed_version, source_sha256, field_name,
  count(*) as snapshot_rows,
  count_if(field_value is null) as null_count,
  count_if(field_value = '') as empty_count,
  count_if(field_value <> '' and field_value rlike r'(?U)^\s*$') as whitespace_only_count,
  count_if(field_value is not null and not (field_value rlike r'(?U)^\s*$')) as non_blank_count,
  count(distinct case when not (field_value rlike r'(?U)^\s*$') then field_value end) as distinct_non_blank_count,
  count_if(field_value is not null and regexp_replace(field_value, r'(?U)^\s+|\s+$', '') <> field_value) as edge_whitespace_count
from cells
group by source_feed_version, source_sha256, field_name
order by source_feed_version, field_name;


## Bloque 6 — ¿Hay problemas de identidad, coordenadas o jerarquía?

Este es el bloque más técnico; puedo estudiar primero el resumen del informe. Comprueba IDs, copias, conversiones, rangos globales y relaciones con padres del mismo snapshot. Los IDs ya resultaron únicos en el bloque 2: ese dato evita multiplicar filas al unir cada hijo con su padre. Si en otro snapshot hubiera duplicados, revisaré primero esa ambigüedad antes de reutilizar la agregación de jerarquía. Los casts y el valor efectivo de location_type son diagnósticos en memoria; no transforman Bronze ni publican Silver. Las 750 referencias de 2026 encuentran padres válidos; los controles de error de ambos snapshots dieron cero. Los rangos globales no prueban pertenencia al límite geográfico de Bogotá.


In [ ]:
%sql
-- BLOQUE 6 — ¿HAY PROBLEMAS DE IDENTIDAD, COORDENADAS O JERARQUÍA? — Consulta de lectura ejecutada mediante Genie.
with base as (
  select source_feed_version, source_sha256, stop_id, stop_code, stop_name, stop_lat, stop_lon, location_type, parent_station, zone_id, wheelchair_boarding, source_file_path, bronze_loaded_at_utc
  from workspace.gtfs_bogota.bronze_gtfs_stops
  where (source_feed_version = '2025-10-28' and source_sha256 = '07ba2ba9bb3b5bc1e38357355956757ce84fc68e6d05b74577bf017e189ab952')
    or (source_feed_version = '2026-09-29' and source_sha256 = '08d14e0b3a5b9caa5451b722659871fd554e3273b94bdd770fdf81a26bf02097')
), typed as (
  select source_feed_version, source_sha256, stop_id, stop_name,
    stop_lat, stop_lon, location_type, parent_station, wheelchair_boarding,
    source_file_path, bronze_loaded_at_utc,
    case when location_type is null or location_type rlike r'(?U)^\s*$' then 0 else try_cast(location_type as int) end as location_type_effective,
    try_cast(stop_lat as double) as lat,
    try_cast(stop_lon as double) as lon,
    coalesce(parent_station rlike r'(?U)^\s*$', true) as blank_parent
  from base
), id_groups as (
  select source_feed_version, source_sha256, stop_id,
    count(*) as copies, count(distinct struct(stop_id, stop_code, stop_name, stop_lat, stop_lon, location_type, parent_station, zone_id, wheelchair_boarding)) as variants
  from base group by source_feed_version, source_sha256, stop_id
), id_checks as (
  select source_feed_version, source_sha256,
    count_if(copies > 1) as duplicate_id_groups,
    sum(copies - variants) as identical_business_copies,
    count_if(variants > 1) as conflicting_id_groups
  from id_groups group by source_feed_version, source_sha256
), trimmed_ids as (
  select source_feed_version, source_sha256,
    regexp_replace(stop_id, r'(?U)^\s+|\s+$', '') as stripped_id,
    count(distinct stop_id) as original_ids
  from base where stop_id is not null and not (stop_id rlike r'(?U)^\s*$')
  group by source_feed_version, source_sha256, regexp_replace(stop_id, r'(?U)^\s+|\s+$', '')
), collision_checks as (
  select source_feed_version, source_sha256, count_if(original_ids > 1) as trimmed_id_collisions
  from trimmed_ids group by source_feed_version, source_sha256
), profile as (
  select c.source_feed_version, c.source_sha256, count(*) as total_rows,
    count_if(c.stop_id is null or c.stop_id rlike r'(?U)^\s*$') as blank_id_rows,
    count_if(c.location_type is not null and not (c.location_type rlike r'(?U)^\s*$') and not (c.location_type rlike r'^[0-4]$')) as invalid_location_type_rows,
    count_if(c.wheelchair_boarding is not null and not (c.wheelchair_boarding rlike r'(?U)^\s*$') and not (c.wheelchair_boarding rlike r'^[0-2]$')) as invalid_wheelchair_rows,
    count_if(c.location_type_effective in (0,1,2) and (c.stop_name is null or c.stop_name rlike r'(?U)^\s*$')) as required_name_missing_rows,
    count_if(c.location_type_effective in (0,1,2) and (c.stop_lat is null or c.stop_lat rlike r'(?U)^\s*$' or c.stop_lon is null or c.stop_lon rlike r'(?U)^\s*$')) as required_coordinates_missing_rows,
    count_if(c.stop_lat is not null and not (c.stop_lat rlike r'(?U)^\s*$') and (c.lat is null or isnan(c.lat) or not (c.lat between -90 and 90))) as invalid_latitude_rows,
    count_if(c.stop_lon is not null and not (c.stop_lon rlike r'(?U)^\s*$') and (c.lon is null or isnan(c.lon) or not (c.lon between -180 and 180))) as invalid_longitude_rows,
    count_if(c.lat = 0 and c.lon = 0) as zero_coordinate_pairs,
    min(c.lat) as min_latitude, max(c.lat) as max_latitude,
    min(c.lon) as min_longitude, max(c.lon) as max_longitude,
    count_if(not c.blank_parent) as parent_reference_rows,
    count_if(not c.blank_parent and p.stop_id is null) as missing_parent_rows,
    count_if(not c.blank_parent and c.parent_station = c.stop_id) as self_parent_rows,
    count_if(c.location_type_effective = 1 and not c.blank_parent) as forbidden_parent_rows,
    count_if(c.location_type_effective in (2,3,4) and c.blank_parent) as required_parent_missing_rows,
    count_if(not c.blank_parent and p.stop_id is not null and (
      (c.location_type_effective in (0,2,3) and not (p.location_type_effective <=> 1)) or
      (c.location_type_effective = 4 and not (p.location_type_effective <=> 0))
    )) as wrong_parent_type_rows,
    count_if(c.bronze_loaded_at_utc is null) as missing_bronze_timestamp_rows,
    count_if(c.source_file_path is null or not (c.source_file_path like concat('%/feed_version=', c.source_feed_version, '/sha256=', c.source_sha256, '/stops.txt'))) as inconsistent_source_path_rows,
    count_if(c.stop_name rlike r'[\p{Cc}]') as name_control_character_rows
  from typed c left join typed p
    on c.source_feed_version = p.source_feed_version and c.source_sha256 = p.source_sha256 and c.parent_station = p.stop_id
  group by c.source_feed_version, c.source_sha256
)
select p.*, d.duplicate_id_groups, d.identical_business_copies, d.conflicting_id_groups, t.trimmed_id_collisions
from profile p
join id_checks d using (source_feed_version, source_sha256)
join collision_checks t using (source_feed_version, source_sha256)
order by source_feed_version;


## Qué haré después de revisar este perfil

Después de revisar este perfil definí stops_v1 con tipos, obligatoriedad y tratamiento de errores. La clave propuesta conserva `(source_feed_version, source_sha256, stop_id)`; no homologaré ubicaciones entre años por un nombre parecido.

Acordé limpiar solo espacios Unicode de los bordes de nombres y conservar opcionales ausentes. Implementé el procesador y amplié el pipeline local; sus pruebas y el despliegue se registran en la guía Silver. Este cuaderno conserva consultas del perfil fuente, no demuestra una publicación remota.

La referencia para las reglas de ubicaciones y jerarquía es [GTFS stops.txt](https://gtfs.org/documentation/schedule/reference/#stopstxt). Los contratos y procedimientos completos siguen en docs/; este perfil no crea una relación directa de stops con routes.
